## Retrieval Pipeline

We need our embeddings also, because we need to convert the user query to embeddings so that our vector DB can match it with stored embeddings

In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-small-en-v1.5",
    model_kwargs = {
        "device" : "cpu" # "cuda" for gpu
    },
    encode_kwargs = {
        "normalize_embeddings" : True,
        "batch_size" : 32
    }
)

d:\Namit Kumar\Learning\Agentic AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1497.20it/s]


In [32]:
Query = "What is Python in very detail" # Query of the user

### 1. Retrieval with Local Vector DB (FAISS)

- Load the stored Vector DB<br>
We use "load_local()" function to load previous stored vector DB

In [2]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "faiss_db",  # Path of the local vector storage DB
    embeddings=embeddings, # Used for converting user query to embeddings
    allow_dangerous_deserialization=True # Here we know our vector store contains index.pkl which contains Document, metadata, etc. Now loading the pickle file can execute arbitrary python code which might be malicious, so langchain requires to explicitly acknowledge this risk by the user.
)

C:\Users\NAMAN KUMAR\AppData\Local\Temp\ipykernel_21964\184905910.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### 2. Create retriever

as_retriever() will convert vectorstore into a retriever. This is because LangChain chains and Agents are built around as_retriever() so different vector databases can be used changeably

In [3]:
retriever = vectorstore.as_retriever(
    search_type = "similarity", # The type of search we are performing.
    search_kwargs = {
        "k" : 4
    }
)

Common options for search_type are :

1. "similarity"
    - Retrieve the nearest vectors using whatever the similarity/distance metric the underlying database/index is configured to use. As we are using a BGE model and {"normalize_embeddings" : True} in HuggingFaceEmbeddings then this will use cosine similarity because our vectors are unit vectors and BGE model prefers cosine similarity or basically a dot product

2. "mmr"
    - Maximum Marginal Relevance. Useful when we dont want several nearly identical chunks

3. "similarity_score_threshold"
    - Returns only chunks whose similarity score is above a threshold


search_kwargs - This is a dictionary of settings passed to search algorithm, different search type supports different keyword arguements.
The {}"k" : 4} means return top 4 most similar chunks or top 4 matching chunks.

### 3. Invoking the retriever

This actually performs retrieval

In [33]:
retrieved_documents = retriever.invoke(Query)

In [34]:
retrieved_documents

[Document(id='36328b05-65be-4e8f-acd6-98301fccb4b4', metadata={'source': 'Data\\Text_Files\\python.txt'}, page_content='Python is a high-level, interpreted, and general-purpose programming language renowned for its exceptional code readability and simplicity. Created by Guido van Rossum and first released in 1991, its core design philosophy prioritizes clean, human-like syntax over complex formatting symbols. By replacing standard curly brackets with mandatory structural indentation, it forces developers to write naturally clean and highly scannable code blocks.Key FeaturesBeginner-Friendly Syntax: Code closely'),
 Document(id='4c11079d-7b32-403d-98d6-f4fa53099f49', metadata={'source': 'Data\\Text_Files\\python.txt'}, page_content="utilize script automation to scrape web data, manage local files, and parse spreadsheets efficiently.Ultimately, Python balances incredible computational power with an approachable learning curve. This unique combination secures its place as one of the world

No we will prepare it to be passed to LLM as context

In [ ]:
context = ""
counter = 1

for document in retrieved_documents:
    context += f"Document {counter}:\n"
    context += document.page_content + "\n"

    source = document.metadata.get("source") or document.metadata.get("producer", "Unknown")

    context += f"Source: {source}\n\n"

    counter += 1

In [36]:
context

'1 Document content : Python is a high-level, interpreted, and general-purpose programming language renowned for its exceptional code readability and simplicity. Created by Guido van Rossum and first released in 1991, its core design philosophy prioritizes clean, human-like syntax over complex formatting symbols. By replacing standard curly brackets with mandatory structural indentation, it forces developers to write naturally clean and highly scannable code blocks.Key FeaturesBeginner-Friendly Syntax: Code closely\nSource of 1 DocumentData\\Text_Files\\python.txt2 Document content : utilize script automation to scrape web data, manage local files, and parse spreadsheets efficiently.Ultimately, Python balances incredible computational power with an approachable learning curve. This unique combination secures its place as one of the world\'s most sought-after programming languages for students, software engineers, and major enterprise corporations alike.\nSource of 2 DocumentData\\Text_

### 4. Passing context + prompt to LLM (Augmentation)

In [37]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(model = "llama-3.1-8b-instant", api_key = os.getenv("GROQ_API_KEY"))

In [25]:
prompt = """"
Answer to the user query preciely according with the given context, dont need to show your thinking in the answer only the answer relevent to user query

Context : {Context}

Query : {Query}

"""

In [48]:
response = llm.invoke(prompt.format(Query = Query, Context=context))
response.content

'Python is a high-level, interpreted, and general-purpose programming language renowned for its exceptional code readability and simplicity. \n\nIt was created by Guido van Rossum and first released in 1991. The core design philosophy prioritizes clean, human-like syntax over complex formatting symbols. \n\nKey Features:\n\n- Beginner-Friendly Syntax: Code closely resembles regular English.\n- Interpreted Execution: Programs execute line-by-line, accelerating rapid prototyping and software debugging cycles.\n- Dynamically Typed: Variable data types are managed automatically at runtime.\n- Cross-Platform Portability: Code runs seamlessly across Windows, macOS, Linux, and specialized hardware like Raspberry Pi.\n- Multi-Paradigm Support: It natively handles object-oriented, procedural, and functional programming frameworks.\n\nIt is often described as "batteries included" due to its massive built-in standard library and highly supportive open-source global community. This deep ecosystem 

### Extra : Using our Cloud based Vector Store DB (Qdrant)

We just have to load our Cloud vectorstore as we loaded our local vectorstore in step 1, rest all steps are same

In [41]:
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
import os

client = QdrantClient(
    api_key=os.getenv("QDRANT_CLUSTER_API_KEY"),
    url=os.getenv("QDRANT_CLUSTER_URL")
)

qdrant_vectorstore = QdrantVectorStore(
    client = client,
    embedding=embeddings,
    collection_name="Agentic_AI_RAG_Course"
)

In [42]:
qdrant_retriever = qdrant_vectorstore.as_retriever(
    search_type = "similarity",
    search_kwargs = {
        "k" : 5
    }
)

In [44]:
qdrant_retrieved_docs = qdrant_retriever.invoke(Query)

In [49]:
Context = ""
counter = 1

for document in qdrant_retrieved_docs:
    context += f"{counter} Document Content : " + document.page_content + "\n"
    context += f"Source of {counter} Document : " + document.metadata['source'] if document.metadata['source'] else document.metadata['producer']
    counter += 1


Context

''

In [50]:
prompt = """"
Answer to the user query preciely according with the given context, dont need to show your thinking in the answer only the answer relevent to user query

Context : {Context}

Query : {Query}

"""

In [51]:
response = llm.invoke(prompt.format(Query = Query, Context = Context))
response.content

"**Introduction to Python**\n\nPython is a high-level, interpreted programming language that was created in the late 1980s by Guido van Rossum. It is a general-purpose language, meaning it can be used for a wide range of applications, including web development, scientific computing, data analysis, artificial intelligence, and more.\n\n**Key Features of Python**\n\n1. **Easy to Learn**: Python has a simple syntax and is relatively easy to read and write, making it a great language for beginners.\n2. **High-Level Language**: Python is a high-level language, meaning it abstracts away many low-level details, allowing developers to focus on the logic of their program without worrying about memory management and other details.\n3. **Interpreted Language**: Python code is interpreted line by line, rather than compiled all at once, which makes it easier to debug and test.\n4. **Object-Oriented**: Python is an object-oriented language, meaning it organizes code into objects that contain data an